# NIDS Feature Selection Cuckoo Search

Cuckoo search feature selection on a synthetic intrusion classification dataset. Synthetic data keeps the notebook self-contained; edit its size, informative feature count, and search parameters. The score uses held-out nearest-centroid accuracy minus a small subset-size penalty.

Run the code cell to generate **`nids-feature-selection-cuckoo-search.png`** in the repository folder.

In [ ]:
# Editable inputs
SAMPLES,FEATURES,INFORMATIVE=240,18,5
POPULATION,ITERATIONS,SEED=28,100,21
MIN_FEATURES,MAX_FEATURES=1,10
OUTPUT="nids-feature-selection-cuckoo-search.png"
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
if not (1<=INFORMATIVE<=FEATURES): raise ValueError("INFORMATIVE must be between 1 and FEATURES")
y=np.repeat([0,1],SAMPLES//2); y=np.r_[y,np.zeros(SAMPLES-len(y),dtype=int)]; X=rng.normal(size=(SAMPLES,FEATURES)); X[:,:INFORMATIVE]+=(2*y-1)[:,None]*1.1; ix=rng.permutation(SAMPLES); tr,te=ix[:int(.75*SAMPLES)],ix[int(.75*SAMPLES):]
def fit(m):
 ids=np.flatnonzero(m)
 if not len(ids): return 0.
 means=np.array([X[tr[y[tr]==c]][:,ids].mean(0) for c in (0,1)]); pred=np.argmin(((X[te][:,ids][:,None,:]-means[None,:,:])**2).sum(2),axis=1); return float(np.mean(pred==y[te])-.02*len(ids)/FEATURES)
def repair(m):
 m=m.copy(); lo=max(1,MIN_FEATURES); hi=min(FEATURES,MAX_FEATURES)
 while m.sum()<lo:m[rng.integers(FEATURES)]=1
 if m.sum()>hi: m[rng.choice(np.flatnonzero(m),m.sum()-hi,False)]=0
 return m.astype(bool)
nests=np.array([repair(rng.random(FEATURES)<.3) for _ in range(POPULATION)]); scores=np.array([fit(m) for m in nests]); best=nests[scores.argmax()].copy(); hist=[]
for _ in range(ITERATIONS):
 for i in range(POPULATION):
  cand=nests[i].copy(); k=min(FEATURES,max(1,int(abs(rng.standard_cauchy())*2))); cand[rng.choice(FEATURES,k,False)]^=True; cand=repair(cand); j=rng.integers(POPULATION); val=fit(cand)
  if val>scores[j]:nests[j]=cand;scores[j]=val
 for j in np.flatnonzero(rng.random(POPULATION)<.25):nests[j]=repair(rng.random(FEATURES)<.3);scores[j]=fit(nests[j])
 if scores.max()>fit(best):best=nests[scores.argmax()].copy()
 hist.append(fit(best))
print("Selected feature indices:",np.flatnonzero(best).tolist()," adjusted score:",round(fit(best),3))
fig,(a,b)=plt.subplots(1,2,figsize=(11,4)); a.bar(range(FEATURES),best,color=["tomato" if i<INFORMATIVE else "steelblue" for i in range(FEATURES)]); a.set(title="Selected features (informative in red)",xlabel="Feature",ylabel="Selected"); b.plot(hist); b.set(title="Best adjusted score",xlabel="Iteration",ylabel="Score"); b.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUTPUT,dpi=160); plt.show(); print("Saved",OUTPUT)